In [3]:
import pandas as pd
from pathlib import Path

series_df = pd.read_csv("../Data/prem_division_series_results.csv")


# CONVERT SERIES RESULTS TO 1 / -1 / 0

def get_series_result(row):
    if row["result"] == "TIE":
        return 0
    return 1 if row["result"] == row["team1"] else -1


series_df["result"] = series_df.apply(get_series_result, axis=1)

series_df = series_df[
    [
        "season",
        "league",
        "division",
        "team1",
        "team2",
        "result",
        "wins1",
        "wins2",
        "ties",
        "games_played",
    ]
]


# GET EACH TEAM'S SERIES SCORE
# Series win = 1, series tie = 0.5, series loss = 0

team1 = series_df[
    ["season", "league", "division", "team1", "result"]
].copy()

team1 = team1.rename(columns={"team1": "team"})

team1["wins"] = (
    (team1["result"] == 1).astype(float)
    + 0.5 * (team1["result"] == 0).astype(float)
)

team2 = series_df[
    ["season", "league", "division", "team2", "result"]
].copy()

team2 = team2.rename(columns={"team2": "team"})

team2["wins"] = (
    (team2["result"] == -1).astype(float)
    + 0.5 * (team2["result"] == 0).astype(float)
)

score_sequences = pd.concat(
    [team1, team2],
    ignore_index=True
)

score_sequences = (
    score_sequences
    .groupby(
        ["season", "league", "division", "team"],
        as_index=False
    )["wins"]
    .sum()
)

score_sequences = score_sequences.sort_values(
    ["season", "division", "wins"]
)


# LOOK UP EACH TEAM'S SCORE

score_lookup = score_sequences.set_index(
    ["season", "division", "team"]
)["wins"].to_dict()


# STRICT UPSETS
# A lower-scoring team beats a higher-scoring team.
# Tied series and equal-score teams do not count.

def is_upset(row):
    if row["result"] == 0:
        return 0

    if row["result"] == 1:
        winner = row["team1"]
        loser = row["team2"]
    else:
        winner = row["team2"]
        loser = row["team1"]

    winner_score = score_lookup[
        (row["season"], row["division"], winner)
    ]

    loser_score = score_lookup[
        (row["season"], row["division"], loser)
    ]

    return int(winner_score < loser_score)


series_df["upset"] = series_df.apply(
    is_upset,
    axis=1
)


# WEAK UPSETS
# Same as strict, plus 0.5 for a tied series between
# teams with different total scores.

def is_weak_upset(row):
    if row["result"] == 0:
        s1 = score_lookup[
            (row["season"], row["division"], row["team1"])
        ]
        s2 = score_lookup[
            (row["season"], row["division"], row["team2"])
        ]
        return 0.5 if s1 != s2 else 0

    return row["upset"]


series_df["upset_weak"] = series_df.apply(
    is_weak_upset,
    axis=1
)

upset_counts = (
    series_df
    .groupby(
        ["season", "league", "division"],
        as_index=False
    )[["upset", "upset_weak"]]
    .sum()
)


# SORT TEAMS BY DECREASING SCORE, AS IN THE MLB OUTPUT

score_sequences = score_sequences.sort_values(
    ["season", "division", "wins"],
    ascending=[True, True, False]
)


# COLLAPSE INTO ONE ROW PER SEASON / DIVISION

scoring_summary = (
    score_sequences
    .groupby(
        ["season", "league", "division"],
        as_index=False
    )
    .agg(
        num_teams=("team", "size"),
        teams=("team", tuple),
        scoring_sequence=("wins", tuple)
    )
)


# GET THE NUMBER OF GAMES BETWEEN EACH PAIR

games_per_pair = (
    series_df
    .groupby(
        ["season", "division"],
        as_index=False
    )["games_played"]
    .first()
    .rename(columns={"games_played": "games_per_pair"})
)


# MERGE THE SUMMARIES

analysis_df = (
    scoring_summary
    .merge(
        games_per_pair,
        on=["season", "division"],
        how="left"
    )
    .merge(
        upset_counts,
        on=["season", "league", "division"],
        how="left"
    )
)

analysis_df = analysis_df.rename(
    columns={"upset": "upset_strict"}
)

analysis_df = analysis_df[
    [
        "season",
        "division",
        "num_teams",
        "games_per_pair",
        "teams",
        "scoring_sequence",
        "upset_strict",
        "upset_weak",
    ]
]

In [4]:
from collections import Counter
from math import comb

# "Games" here means head-to-head series: one per pair of teams.
analysis_df["num_games"] = analysis_df["num_teams"].apply(
    lambda n: comb(int(n), 2)
)

# Exclude pairs whose teams have equal total scores.
analysis_df["num_admissible"] = analysis_df["num_games"] - (
    analysis_df["scoring_sequence"].apply(
        lambda scores: sum(
            comb(count, 2)
            for count in Counter(scores).values()
        )
    )
)

# Zero admissible games means the frequency is undefined.
# These values will appear as blank cells in the exported CSV.
denominator = analysis_df["num_admissible"].astype(float)
denominator = denominator.where(denominator > 0)

analysis_df["upset_strict_frequency"] = (
    analysis_df["upset_strict"] / denominator
)

analysis_df["upset_weak_frequency"] = (
    analysis_df["upset_weak"] / denominator
)

In [5]:
# SAVE THE OUTPUT

output_path = Path("../Analysis/premier_league_analysis.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

analysis_df.to_csv(output_path, index=False)